# Review FGTV & ENTC — uganda.csv vs raw output.file
Compares post-adjustment `uganda.csv` against the raw SISEPUEDE output `c585e7e9-e32f-4131-999b-ee7fc5ec014e.csv` for years 2019, 2030, 2040, 2050, 2070, by strategy.

In [1]:
import pandas as pd
import numpy as np

RUN = 'sisepuede_summary_results_run_sisepuede_run_2026-05-10T13;25;55.848572'
DIR = f'../ssp_run_output/{RUN}/'

BASE_YEAR = 2015
REVIEW_YEARS = [2019, 2030, 2040, 2050, 2070]
REVIEW_TPS   = [y - BASE_YEAR for y in REVIEW_YEARS]  # [4, 15, 25, 35, 55]

uga = pd.read_csv(DIR + 'uganda.csv')
raw = pd.read_csv(DIR + 'c585e7e9-e32f-4131-999b-ee7fc5ec014e.csv')

# year label for readability
uga['year'] = uga['time_period'] + BASE_YEAR
raw['year'] = raw['time_period'] + BASE_YEAR

print(f'uganda.csv   : {uga.shape}')
print(f'raw output   : {raw.shape}')
print(f'Strategies in uganda.csv: {sorted(uga.primary_id.unique())}')

uganda.csv   : (3224, 4120)
raw output   : (3472, 4120)
Strategies in uganda.csv: [np.int64(0), np.int64(1001), np.int64(2002), np.int64(3003), np.int64(4004), np.int64(5005), np.int64(6006), np.int64(7007), np.int64(8008), np.int64(9009), np.int64(10010), np.int64(11011), np.int64(12012), np.int64(13013), np.int64(14014), np.int64(15015), np.int64(16016), np.int64(17017), np.int64(18018), np.int64(19019), np.int64(20020), np.int64(21021), np.int64(22022), np.int64(23023), np.int64(24024), np.int64(25025), np.int64(26026), np.int64(27027), np.int64(28028), np.int64(29029), np.int64(30030), np.int64(31031), np.int64(32032), np.int64(33033), np.int64(34034), np.int64(35035), np.int64(36036), np.int64(37037), np.int64(38038), np.int64(39039), np.int64(40040), np.int64(41041), np.int64(42042), np.int64(43043), np.int64(44044), np.int64(45045), np.int64(46046), np.int64(47047), np.int64(48048), np.int64(49049), np.int64(50050), np.int64(51051), np.int64(52052), np.int64(53053), np.int64(540

## 1. Subsector totals — high-level view

In [2]:
TOTAL_COLS = ['emission_co2e_subsector_total_fgtv', 'emission_co2e_subsector_total_entc']

uga_totals = (
    uga[uga['time_period'].isin(REVIEW_TPS)]
    [['primary_id', 'year'] + TOTAL_COLS]
    .copy()
)
raw_totals = (
    raw[raw['time_period'].isin(REVIEW_TPS)]
    [['primary_id', 'year'] + [c for c in TOTAL_COLS if c in raw.columns]]
    .copy()
)

merged_totals = uga_totals.merge(
    raw_totals,
    on=['primary_id', 'year'],
    suffixes=('_adj', '_raw')
)

for sec in ['fgtv', 'entc']:
    col = f'emission_co2e_subsector_total_{sec}'
    adj_col = col + '_adj' if col + '_adj' in merged_totals.columns else col
    raw_col = col + '_raw' if col + '_raw' in merged_totals.columns else col
    if adj_col not in merged_totals.columns or raw_col not in merged_totals.columns:
        print(f'{sec.upper()} subsector total not found in raw — skipping')
        continue
    display_df = merged_totals[['primary_id', 'year', adj_col, raw_col]].copy()
    display_df['diff'] = display_df[adj_col] - display_df[raw_col]
    display_df.columns = ['strategy', 'year', f'{sec.upper()} adjusted', f'{sec.upper()} raw', 'diff (adj - raw)']
    print(f'\n=== {sec.upper()} subsector total ===')
    display(display_df.sort_values(['strategy', 'year']).reset_index(drop=True))


=== FGTV subsector total ===


,strategy,year,FGTV adjusted,FGTV raw,diff (adj - raw)
0,0,2019,0.000025,0.000005,1.993892e-05
1,0,2030,0.020593,0.027189,-6.596494e-03
2,0,2040,0.041157,0.067616,-2.645926e-02
3,0,2050,0.061721,0.106102,-4.438011e-02
4,0,2070,0.102850,0.102850,-4.996004e-16
...,...,...,...,...,...
305,61061,2019,0.000025,0.000005,1.993892e-05
306,61061,2030,0.006849,0.026907,-2.005793e-02
307,61061,2040,0.013670,0.048179,-3.450921e-02
308,61061,2050,0.020490,0.056443,-3.595271e-02



=== ENTC subsector total ===


,strategy,year,ENTC adjusted,ENTC raw,diff (adj - raw)
0,0,2019,0.330493,0.137722,0.192772
1,0,2030,1.941939,1.675990,0.265950
2,0,2040,6.406034,7.497674,-1.091640
3,0,2050,10.870129,11.540776,-0.670648
4,0,2070,19.798318,10.755190,9.043127
...,...,...,...,...,...
305,61061,2019,0.330493,0.137722,0.192772
306,61061,2030,0.620138,1.599382,-0.979244
307,61061,2040,0.710229,4.458186,-3.747957
308,61061,2050,0.800321,4.055293,-3.254972


## 2. FGTV — detail by column

In [3]:
fgtv_cols = [c for c in uga.columns if '_fgtv_' in c and 'subsector_total' not in c]
print(f'FGTV detail columns: {len(fgtv_cols)}')

uga_fgtv = uga[uga['time_period'].isin(REVIEW_TPS)][['primary_id', 'year'] + fgtv_cols].copy()
raw_fgtv = raw[raw['time_period'].isin(REVIEW_TPS)][['primary_id', 'year'] + [c for c in fgtv_cols if c in raw.columns]].copy()

# Melt to long format for easy comparison
uga_fgtv_long = uga_fgtv.melt(id_vars=['primary_id', 'year'], var_name='variable', value_name='adjusted')
raw_fgtv_long = raw_fgtv.melt(id_vars=['primary_id', 'year'], var_name='variable', value_name='raw')

fgtv_cmp = uga_fgtv_long.merge(raw_fgtv_long, on=['primary_id', 'year', 'variable'])
fgtv_cmp['diff'] = fgtv_cmp['adjusted'] - fgtv_cmp['raw']
fgtv_cmp['pct_diff'] = np.where(
    fgtv_cmp['raw'].abs() > 1e-10,
    (fgtv_cmp['diff'] / fgtv_cmp['raw'].abs()) * 100,
    np.nan
)

# Show only rows where there's a non-trivial difference
fgtv_diff = fgtv_cmp[fgtv_cmp['diff'].abs() > 1e-6].sort_values(['primary_id', 'year', 'variable'])
print(f'Rows with non-trivial diff: {len(fgtv_diff)}')
display(fgtv_diff.reset_index(drop=True))

FGTV detail columns: 103
Rows with non-trivial diff: 3203


,primary_id,year,variable,adjusted,raw,diff,pct_diff
0,0,2019,emission_co2e_ch4_fgtv_dtp_fuel_natural_gas,0.000024,0.000005,0.000019,400.000393
1,0,2030,emission_co2e_ch4_fgtv_dtp_fuel_crude,0.000029,0.004628,-0.004599,-99.375519
2,0,2030,emission_co2e_ch4_fgtv_dtp_fuel_natural_gas,0.010325,0.002254,0.008071,358.067941
3,0,2030,emission_co2e_ch4_fgtv_dtp_fuel_oil,0.000008,0.000009,-0.000001,-13.079101
4,0,2030,emission_co2e_ch4_fgtv_flaring_fuel_natural_gas,0.000177,0.000000,0.000177,NaN
...,...,...,...,...,...,...,...
3198,61061,2050,emission_nongas_fgtv_kt_nmvoc_dtp_fuel_natural...,1.864261,3.943315,-2.079054,-52.723501
3199,61061,2050,emission_nongas_fgtv_kt_nmvoc_dtp_fuel_oil,0.040272,0.205099,-0.164828,-80.364881
3200,61061,2050,emission_nongas_fgtv_kt_nmvoc_flaring_fuel_nat...,0.008641,0.018113,-0.009473,-52.296180
3201,61061,2050,emission_nongas_fgtv_kt_nmvoc_venting_fuel_crude,0.037100,0.273953,-0.236852,-86.457461


In [4]:
# Pivot: adjusted vs raw per year, for each strategy — summed across all FGTV columns
fgtv_sum = fgtv_cmp.groupby(['primary_id', 'year'])[['adjusted', 'raw', 'diff']].sum().reset_index()
fgtv_sum.columns = ['strategy', 'year', 'FGTV total (adjusted)', 'FGTV total (raw)', 'diff (adj - raw)']
print('\nFGTV — sum of all detail columns by strategy & year')
display(fgtv_sum.sort_values(['strategy', 'year']).reset_index(drop=True))


FGTV — sum of all detail columns by strategy & year


,strategy,year,FGTV total (adjusted),FGTV total (raw),diff (adj - raw)
0,0,2019,0.178570,0.178550,1.993892e-05
1,0,2030,2.107021,112.670127,-1.105631e+02
2,0,2040,4.035198,10.965216,-6.930019e+00
3,0,2050,5.963374,11.970483,-6.007108e+00
4,0,2070,9.819728,9.819728,2.997529e-15
...,...,...,...,...,...
305,61061,2019,0.178570,0.178550,1.993892e-05
306,61061,2030,0.931234,112.078683,-1.111474e+02
307,61061,2040,1.683624,7.423317,-5.739693e+00
308,61061,2050,2.436014,6.782330,-4.346316e+00


## 3. ENTC — detail by column

In [5]:
entc_cols = [c for c in uga.columns if '_entc_' in c and 'subsector_total' not in c]
print(f'ENTC detail columns: {len(entc_cols)}')

uga_entc = uga[uga['time_period'].isin(REVIEW_TPS)][['primary_id', 'year'] + entc_cols].copy()
raw_entc = raw[raw['time_period'].isin(REVIEW_TPS)][['primary_id', 'year'] + [c for c in entc_cols if c in raw.columns]].copy()

uga_entc_long = uga_entc.melt(id_vars=['primary_id', 'year'], var_name='variable', value_name='adjusted')
raw_entc_long = raw_entc.melt(id_vars=['primary_id', 'year'], var_name='variable', value_name='raw')

entc_cmp = uga_entc_long.merge(raw_entc_long, on=['primary_id', 'year', 'variable'])
entc_cmp['diff'] = entc_cmp['adjusted'] - entc_cmp['raw']
entc_cmp['pct_diff'] = np.where(
    entc_cmp['raw'].abs() > 1e-10,
    (entc_cmp['diff'] / entc_cmp['raw'].abs()) * 100,
    np.nan
)

# Sum across all ENTC columns per strategy & year
entc_sum = entc_cmp.groupby(['primary_id', 'year'])[['adjusted', 'raw', 'diff']].sum().reset_index()
entc_sum.columns = ['strategy', 'year', 'ENTC total (adjusted)', 'ENTC total (raw)', 'diff (adj - raw)']
print('\nENTC — sum of all detail columns by strategy & year')
display(entc_sum.sort_values(['strategy', 'year']).reset_index(drop=True))

ENTC detail columns: 526

ENTC — sum of all detail columns by strategy & year


,strategy,year,ENTC total (adjusted),ENTC total (raw),diff (adj - raw)
0,0,2019,4.000001e+10,4.000001e+10,5.505021e-02
1,0,2030,4.000001e+10,4.000001e+10,2.581860e+02
2,0,2040,4.000002e+10,4.000002e+10,3.317229e+02
3,0,2050,4.000002e+10,4.000002e+10,1.856629e+03
4,0,2070,4.000004e+10,4.000004e+10,2.238636e-11
...,...,...,...,...,...
305,61061,2019,4.000001e+10,4.000001e+10,5.505021e-02
306,61061,2030,4.000001e+10,4.000001e+10,-3.674723e+02
307,61061,2040,4.000001e+10,4.000001e+10,2.419071e+02
308,61061,2050,4.000002e+10,4.000002e+10,1.451339e+03


In [6]:
# Show only rows with non-trivial differences
entc_diff = entc_cmp[entc_cmp['diff'].abs() > 1e-6].sort_values(['primary_id', 'year', 'variable'])
print(f'ENTC rows with non-trivial diff: {len(entc_diff)}')
display(entc_diff.reset_index(drop=True))

ENTC rows with non-trivial diff: 24272


,primary_id,year,variable,adjusted,raw,diff,pct_diff
0,0,2019,emission_co2e_ch4_entc_generation_pp_biomass,0.000007,0.008642,-0.008636,-99.923169
1,0,2019,emission_co2e_ch4_entc_generation_pp_coal,0.000008,0.000000,0.000008,NaN
2,0,2019,emission_co2e_ch4_entc_generation_pp_coal_ccs,0.000008,0.000000,0.000008,NaN
3,0,2019,emission_co2e_ch4_entc_generation_pp_gas,0.000008,0.000000,0.000008,NaN
4,0,2019,emission_co2e_ch4_entc_generation_pp_gas_ccs,0.000008,0.000000,0.000008,NaN
...,...,...,...,...,...,...,...
24267,61061,2050,totalvalue_enfu_fuel_consumed_entc_fuel_gasoline,12.300021,28.035672,-15.735651,-56.127246
24268,61061,2050,totalvalue_enfu_fuel_consumed_entc_fuel_natura...,6.167224,382.117390,-375.950166,-98.386039
24269,61061,2050,totalvalue_enfu_fuel_consumed_entc_fuel_nuclear,406.397163,1197.314997,-790.917835,-66.057624
24270,61061,2050,totalvalue_enfu_fuel_consumed_entc_fuel_oil,3.890965,61.286452,-57.395488,-93.651183


## 4. Ramp check — ENTC trajectory for one strategy (primary_id = 0)

In [8]:
# Check the full time series for primary_id=0 to verify the ramp shape
pid_check = uga['primary_id'].max()  # baseline strategy

uga_ts = uga[uga['primary_id'] == pid_check][['year', 'time_period'] + entc_cols + ['emission_co2e_subsector_total_entc']].copy()
raw_ts = raw[raw['primary_id'] == pid_check][['year', 'time_period'] + [c for c in entc_cols if c in raw.columns]].copy()

uga_ts['ENTC_total_adj'] = uga_ts[entc_cols].sum(axis=1)
raw_ts['ENTC_total_raw'] = raw_ts[[c for c in entc_cols if c in raw.columns]].sum(axis=1)

ts_cmp = uga_ts[['year', 'ENTC_total_adj', 'emission_co2e_subsector_total_entc']].merge(
    raw_ts[['year', 'ENTC_total_raw']], on='year'
)

print(f'\nENTC full time series — strategy {pid_check}')
display(ts_cmp.sort_values('year').reset_index(drop=True))


ENTC full time series — strategy 61061


,year,ENTC_total_adj,emission_co2e_subsector_total_entc,ENTC_total_raw
0,2019,4.000001e+10,0.330493,4.000001e+10
1,2020,4.000001e+10,0.268107,4.000001e+10
2,2021,4.000001e+10,0.272487,4.000001e+10
3,2022,4.000001e+10,0.299754,4.000001e+10
4,2023,4.000001e+10,0.409098,4.000001e+10
5,2024,4.000001e+10,0.383866,4.000001e+10
6,2025,4.000001e+10,0.427917,4.000001e+10
7,2026,4.000001e+10,0.492003,4.000001e+10
8,2027,4.000001e+10,0.593110,4.000001e+10
9,2028,4.000001e+10,0.602120,4.000001e+10


In [9]:
# Same check for FGTV
uga_fgtv_ts = uga[uga['primary_id'] == pid_check][['year', 'time_period'] + fgtv_cols + ['emission_co2e_subsector_total_fgtv']].copy()
raw_fgtv_ts = raw[raw['primary_id'] == pid_check][['year', 'time_period'] + [c for c in fgtv_cols if c in raw.columns]].copy()

uga_fgtv_ts['FGTV_total_adj'] = uga_fgtv_ts[fgtv_cols].sum(axis=1)
raw_fgtv_ts['FGTV_total_raw'] = raw_fgtv_ts[[c for c in fgtv_cols if c in raw.columns]].sum(axis=1)

fgtv_ts_cmp = uga_fgtv_ts[['year', 'FGTV_total_adj', 'emission_co2e_subsector_total_fgtv']].merge(
    raw_fgtv_ts[['year', 'FGTV_total_raw']], on='year'
)

print(f'\nFGTV full time series — strategy {pid_check}')
display(fgtv_ts_cmp.sort_values('year').reset_index(drop=True))


FGTV full time series — strategy 61061


,year,FGTV_total_adj,emission_co2e_subsector_total_fgtv,FGTV_total_raw
0,2019,0.178570,0.000025,0.178550
1,2020,0.178844,0.000028,0.178822
2,2021,0.254083,0.000710,0.178963
3,2022,0.329322,0.001392,0.178998
4,2023,0.404561,0.002074,0.179334
5,2024,0.479800,0.002756,0.179282
6,2025,0.555039,0.003439,0.180172
7,2026,0.630278,0.004121,0.181326
8,2027,0.705517,0.004803,6.074664
9,2028,0.780756,0.005485,60.009752
